In [1]:
import sys
import torch

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Selected device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

Python version: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
PyTorch version: 2.7.1+cu128
CUDA available: True
Selected device: cuda
GPU: NVIDIA GeForce GTX 1650
CUDA version: 12.8


In [2]:
pairs = [
    ("hello", "bonjour"),
    ("thank you", "merci"),
    ("good morning", "bonjour"),
    ("good night", "bonne nuit"),
    ("i am happy", "je suis heureux"),
    ("i am sad", "je suis triste"),
    ("how are you", "comment allez vous"),
    ("i love you", "je t aime"),
    ("see you soon", "a bientot"),
    ("goodbye", "au revoir"),
]

In [3]:
# Special tokens help us handle sequence boundaries and unknown words.
PAD = "<PAD>"  # Padding token: makes sequences equal in length
SOS = "<SOS>"  # Start-of-sequence token: tells the decoder to begin
EOS = "<EOS>"  # End-of-sequence token: tells the decoder when to stop
UNK = "<UNK>"  # Unknown token: handles words missing from the vocabulary


def build_vocab(sentences):
    # Collect every unique word from the supplied sentences.
    words = set()

    for sentence in sentences:
        words.update(sentence.split())  # Split each sentence into word tokens

    # Give special tokens fixed IDs, then assign IDs to normal words.
    tokens = [PAD, SOS, EOS, UNK] + sorted(words)

    # Convert words into integer IDs for neural-network input.
    word_to_id = {
        word: idx for idx, word in enumerate(tokens)
    }

    # Allow us to convert predicted IDs back into readable words.
    id_to_word = {
        idx: word for word, idx in word_to_id.items()
    }

    return word_to_id, id_to_word


# Keep source-language and target-language vocabularies separate.
source_sentences = [src for src, tgt in pairs]
target_sentences = [tgt for src, tgt in pairs]

src_vocab, src_id_to_word = build_vocab(source_sentences)
tgt_vocab, tgt_id_to_word = build_vocab(target_sentences)

# Inspect the vocabulary sizes.
print("Source vocabulary size:", len(src_vocab))
print("Target vocabulary size:", len(tgt_vocab))

# Example: convert a sentence into token IDs.
example = "i am happy blah blah blah".split()

# Use <UNK> if a word is not present in the vocabulary.
# here UNK is at index 3 
# so for unkown words it will return 3
example_ids = [
    src_vocab.get(word, src_vocab[UNK])
    for word in example
]

print("Tokens:", example)
print("Token IDs:", example_ids)

Source vocabulary size: 20
Target vocabulary size: 21
Tokens: ['i', 'am', 'happy', 'blah', 'blah', 'blah']
Token IDs: [11, 4, 8, 3, 3, 3]


In [4]:
# Add <SOS> and <EOS> and encode sequences

def encode_sentence(sentence,vocab):
    # Convert each word into its vocabulary ID.
    # If a word is missing, use the <UNK> token ID.
    token_ids = [
        vocab.get(word,vocab[UNK])
        for word in sentence.split()
    ]

    # Add <SOS> at the beginning and <EOS> at the end.
    # This teaches the decoder when to start and stop generating.
    token_ids = [vocab[SOS]] + token_ids + [vocab[EOS]]

    return token_ids

# Encode one source sentence.
source_example = encode_sentence("i am happy", src_vocab)

# Encode its corresponding target sentence.
target_example = encode_sentence("je suis heureux", tgt_vocab)

print("Source tokens:", source_example)
print("Target tokens:", target_example)

# Convert IDs back into words to verify the encoding.
print(
    "Source reconstructed:",
    [src_id_to_word[token_id] for token_id in source_example]
)

print(
    "Target reconstructed:",
    [tgt_id_to_word[token_id] for token_id in target_example]
)

Source tokens: [1, 11, 4, 8, 2]
Target tokens: [1, 13, 17, 12, 2]
Source reconstructed: ['<SOS>', 'i', 'am', 'happy', '<EOS>']
Target reconstructed: ['<SOS>', 'je', 'suis', 'heureux', '<EOS>']


In [5]:
import torch
from torch.nn.utils.rnn import pad_sequence

# Encode all source sentences using the source vocabulary.
source_sequences = [
    encode_sentence(source, src_vocab)
    for source, target in pairs
]

# Encode all target sentences using the target vocabulary.
target_sequences = [
    encode_sentence(target, tgt_vocab)
    for source, target in pairs
]

# Find the <PAD> token ID in each vocabulary.
src_pad_id = src_vocab[PAD]
tgt_pad_id = tgt_vocab[PAD]

# Convert each list of token IDs into a PyTorch tensor.
# Long tensors are required for embedding layers and token-index targets.
source_tensors = [
    torch.tensor(sequence, dtype=torch.long)
    for sequence in source_sequences
]

target_tensors = [
    torch.tensor(sequence, dtype=torch.long)
    for sequence in target_sequences
]

# Pad sequences to the longest sequence in their respective batch.
# batch_first=True gives shape: (batch_size, sequence_length).
source_batch = pad_sequence(
    source_tensors,
    batch_first=True,
    padding_value=src_pad_id
)

target_batch = pad_sequence(
    target_tensors,
    batch_first=True,
    padding_value=tgt_pad_id
)

# Inspect the batch dimensions and the first two examples.
print("Source batch shape:", source_batch.shape)
print("Target batch shape:", target_batch.shape)

print("\nFirst two source sequences:")
print(source_batch[:2])

print("\nFirst two target sequences:")
print(target_batch[:2])

Source batch shape: torch.Size([10, 5])
Target batch shape: torch.Size([10, 5])

First two source sequences:
tensor([[ 1,  9,  2,  0,  0],
        [ 1, 18, 19,  2,  0]])

First two target sequences:
tensor([[ 1,  9,  2,  0,  0],
        [ 1, 14,  2,  0,  0]])


In [6]:
# Create the padding mask

# True means the position contains a real token.
# False means the position is padding.
source_mask = source_batch != src_pad_id
target_mask = target_batch != tgt_pad_id

print("Source mask for first two examples:")
print(source_mask[:2])

print("\nTarget mask for first two examples:")
print(target_mask[:2])

# Count real tokens and padding tokens in the source batch.
print("\nReal source tokens:", source_mask.sum().item())
print("Source padding tokens:", (~source_mask).sum().item())

Source mask for first two examples:
tensor([[ True,  True,  True, False, False],
        [ True,  True,  True,  True, False]])

Target mask for first two examples:
tensor([[ True,  True,  True, False, False],
        [ True,  True,  True, False, False]])

Real source tokens: 43
Source padding tokens: 7


In [7]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# Use the GPU that we verified earlier.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        pad_id
    ):
        super().__init__()

        # Convert each token ID into a trainable dense vector.
        # Input:  (batch_size, sequence_length)
        # Output: (batch_size, sequence_length, embedding_dim)
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_id
        )

        # Process token embeddings in sequence.
        # batch_first=True means input/output shapes start with batch size.
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

    def forward(self, source, source_lengths):
        # Convert token IDs into embeddings.
        embedded = self.embedding(source)

        # Exclude padded positions from the LSTM's sequence processing.
        # Lengths must be CPU integers for pack_padded_sequence.
        packed = pack_padded_sequence(
            embedded,
            source_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        # Process each real sequence without treating padding as real input.
        packed_outputs, (hidden, cell) = self.lstm(packed)

        # Restore outputs to a padded batch for inspection or later use.
        outputs, _ = pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=source.size(1)
        )

        # Return token-level outputs and the final hidden/cell states.
        return outputs, hidden, cell


# Choose small dimensions to fit comfortably on your GTX 1650.
EMBEDDING_DIM = 64
HIDDEN_DIM = 128

# Create the encoder using the source-language vocabulary.
encoder = Encoder(
    vocab_size=len(src_vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    pad_id=src_pad_id
).to(device)

print("Encoder created successfully!")
print("Device:", next(encoder.parameters()).device)

Encoder created successfully!
Device: cuda:0


In [8]:
# Count real tokens in each source sequence.
# Because <SOS> and <EOS> are real tokens, they are included.
source_lengths = source_mask.sum(dim=1)

# Move the input batch to the GPU for model computation.
source_batch_gpu = source_batch.to(device)

# Run a forward pass without calculating gradients.
# This is a shape test, not model training.
encoder.eval()

with torch.no_grad():
    encoder_outputs, hidden, cell = encoder(
        source_batch_gpu,
        source_lengths
    )

# Inspect the dimensions of the returned tensors.
print("Input shape:", source_batch_gpu.shape)
print("Encoder outputs shape:", encoder_outputs.shape)
print("Hidden state shape:", hidden.shape)
print("Cell state shape:", cell.shape)
print("All outputs on GPU:", encoder_outputs.is_cuda)

Input shape: torch.Size([10, 5])
Encoder outputs shape: torch.Size([10, 5, 128])
Hidden state shape: torch.Size([1, 10, 128])
Cell state shape: torch.Size([1, 10, 128])
All outputs on GPU: True


In [9]:
import torch
import torch.nn as nn


class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        pad_id
    ):
        super().__init__()

        # Convert target token IDs into trainable vectors.
        # Example input:  (10, 1)
        # Example output: (10, 1, 64)
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_id
        )

        # Process one target token at a time.
        # The encoder provides the initial hidden and cell states.
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # Convert each 128-dimensional LSTM output into
        # one score for every token in the target vocabulary.
        self.output_layer = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(self, target_token, hidden, cell):
        # target_token shape: (batch_size, 1)
        # Embed the current target token.
        embedded = self.embedding(target_token)
        # Shape: (batch_size, 1, 64)

        # Process the token using the current recurrent states.
        output, (hidden, cell) = self.lstm(
            embedded,
            (hidden, cell)
        )
        # output shape: (batch_size, 1, 128)
        # hidden and cell shape: (1, batch_size, 128)

        # Produce vocabulary scores for the next token.
        logits = self.output_layer(output)
        # Shape: (batch_size, 1, target_vocab_size)

        # Return scores and updated states for the next step.
        return logits, hidden, cell

In [10]:
# Use the same dimensions as the encoder.
# The decoder uses the target-language vocabulary.
decoder = Decoder(
    vocab_size=len(tgt_vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    pad_id=tgt_pad_id
).to(device)

# Start decoding with <SOS> for all 10 examples.
# Shape: (batch_size, 1) = (10, 1)
target_token = torch.full(
    (source_batch.size(0), 1),
    fill_value=tgt_vocab[SOS],
    dtype=torch.long,
    device=device
)

# Test a single decoding step using the encoder's states.
# No training happens here; we're only checking shapes.
decoder.eval()

with torch.no_grad():
    logits, decoder_hidden, decoder_cell = decoder(
        target_token,
        hidden,
        cell
    )

print("Target token shape:", target_token.shape)
print("Logits shape:", logits.shape)
print("Decoder hidden shape:", decoder_hidden.shape)
print("Decoder cell shape:", decoder_cell.shape)
print("Decoder on GPU:", logits.is_cuda)

Target token shape: torch.Size([10, 1])
Logits shape: torch.Size([10, 1, 21])
Decoder hidden shape: torch.Size([1, 10, 128])
Decoder cell shape: torch.Size([1, 10, 128])
Decoder on GPU: True


In [11]:
import torch
import torch.nn as nn

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, sos_id, pad_id):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.sos_id = sos_id
        self.pad_id = pad_id

    def forward(self, source, source_lengths, target):
        # Encode the complete source sentence
        _, hidden, cell = self.encoder(source, source_lengths)

        # Use SOS and the target words as decoder inputs, excluding EOS
        target_input = target[:, :-1]

        all_logits = []

        # Process one target token at a time using teacher forcing
        for t in range(target_input.size(1)):
            token = target_input[:, t].unsqueeze(1)

            # Predict the next token
            logits, hidden, cell = self.decoder(token, hidden, cell)

            # Remove the sequence-length dimension
            all_logits.append(logits.squeeze(1))

        # Shape: (batch_size, target_length - 1, vocabulary_size)
        return torch.stack(all_logits, dim=1)

In [12]:
# Combine the encoder and decoder into one trainable model.
model = Seq2Seq(
    encoder=encoder,
    decoder=decoder,
    sos_id=tgt_vocab[SOS],
    pad_id=tgt_pad_id
).to(device)

print("Seq2Seq model created!")
print("Model device:", next(model.parameters()).device)

Seq2Seq model created!
Model device: cuda:0


In [13]:
# Move the existing batches to your GPU
source_gpu = source_batch.to(device)
target_gpu = target_batch.to(device)

# Calculate the actual source sentence lengths
source_lengths = source_mask.sum(dim=1).cpu()

# Run one forward pass
model.eval()
with torch.no_grad():
    logits = model(source_gpu, source_lengths, target_gpu)

# Expected next tokens exclude the initial SOS
expected = target_gpu[:, 1:]

print("Logits shape:", logits.shape)
print("Expected shape:", expected.shape)
print("Same sequence dimensions:", logits.shape[:2] == expected.shape)

Logits shape: torch.Size([10, 4, 21])
Expected shape: torch.Size([10, 4])
Same sequence dimensions: True


In [14]:
import torch
import torch.nn as nn

# Ignore PAD tokens when calculating the loss
criterion = nn.CrossEntropyLoss(ignore_index=tgt_pad_id)

# Update model weights using Adam
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Move training data to CUDA
source_gpu = source_batch.to(device)
target_gpu = target_batch.to(device)

# pack_padded_sequence expects lengths on the CPU
source_lengths = source_mask.sum(dim=1).cpu()

EPOCHS = 200

for epoch in range(EPOCHS):
    # Enable training behavior
    model.train()

    # Clear gradients from the previous iteration
    optimizer.zero_grad()

    # Predict the next target token at each position
    logits = model(source_gpu, source_lengths, target_gpu)

    # Expected outputs exclude the initial SOS token
    expected = target_gpu[:, 1:]

    # Flatten sequences for cross-entropy
    loss = criterion(
        logits.reshape(-1, len(tgt_vocab)),
        expected.reshape(-1)
    )

    # Calculate gradients
    loss.backward()

    # Prevent excessively large gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    # Update model parameters
    optimizer.step()

    # Display progress
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch + 1}/{EPOCHS}], Loss: {loss.item():.4f}")

Epoch [20/200], Loss: 0.0090
Epoch [40/200], Loss: 0.0005
Epoch [60/200], Loss: 0.0002
Epoch [80/200], Loss: 0.0002
Epoch [100/200], Loss: 0.0002
Epoch [120/200], Loss: 0.0001
Epoch [140/200], Loss: 0.0001
Epoch [160/200], Loss: 0.0001
Epoch [180/200], Loss: 0.0001
Epoch [200/200], Loss: 0.0001


In [15]:
# Greedy decoding — generate translations

def translate(sentence, max_length=10):
    # Switch the model to evaluation mode
    model.eval()

    # Convert the input sentence into token IDs
    source_ids = encode_sentence(sentence.lower(), src_vocab)

    # Add a batch dimension and move input to CUDA
    source = torch.tensor(
        [source_ids], dtype=torch.long, device=device
    )

    # Store the actual source sentence length on CPU
    source_lengths = torch.tensor([len(source_ids)])

    with torch.no_grad():
        # Encode the source sentence
        _, hidden, cell = model.encoder(source, source_lengths)

        # Begin generation with SOS
        token = torch.tensor(
            [[tgt_vocab[SOS]]], dtype=torch.long, device=device
        )

        predicted_ids = []

        for _ in range(max_length):
            # Predict the next token
            logits, hidden, cell = model.decoder(token, hidden, cell)

            # Select the token with the highest probability
            next_id = logits[:, -1, :].argmax(dim=-1).item()

            # Stop when EOS is generated
            if next_id == tgt_vocab[EOS]:
                break

            predicted_ids.append(next_id)

            # Feed the prediction back into the decoder
            token = torch.tensor(
                [[next_id]], dtype=torch.long, device=device
            )

    # Convert predicted IDs back into words
    words = [
        tgt_id_to_word[idx]
        for idx in predicted_ids
        if idx not in (tgt_vocab[SOS], tgt_vocab[PAD])
    ]

    return " ".join(words)

In [16]:
# Test sentences from the training dataset
test_sentences = [
    "hello",
    "thank you",
    "good night",
    "i am happy",
    "i love you",
    "goodbye"
]

for sentence in test_sentences:
    print(f"English: {sentence}")
    print(f"French:  {translate(sentence)}")
    print("-" * 30)

English: hello
French:  bonjour
------------------------------
English: thank you
French:  merci
------------------------------
English: good night
French:  bonne nuit
------------------------------
English: i am happy
French:  je suis heureux
------------------------------
English: i love you
French:  je t aime
------------------------------
English: goodbye
French:  au revoir
------------------------------


In [17]:
import sacrebleu

# Generate translations for all sentence pairs
predictions = [translate(english) for english, french in pairs]
references = [french for english, french in pairs]

# Calculate and display sentence-level BLEU scores
for (english, reference), prediction in zip(pairs, predictions):

    # Calculate BLEU for this individual sentence
    sentence_bleu = sacrebleu.sentence_bleu(
        prediction,
        [reference],
        smooth_method="exp",
        use_effective_order=True
    )

    print(f"English:    {english}")
    print(f"Reference:  {reference}")
    print(f"Predicted:  {prediction}")
    print(f"BLEU Score: {sentence_bleu.score:.2f}/100")
    print("-" * 35)

English:    hello
Reference:  bonjour
Predicted:  bonjour
BLEU Score: 100.00/100
-----------------------------------
English:    thank you
Reference:  merci
Predicted:  merci
BLEU Score: 100.00/100
-----------------------------------
English:    good morning
Reference:  bonjour
Predicted:  bonjour
BLEU Score: 100.00/100
-----------------------------------
English:    good night
Reference:  bonne nuit
Predicted:  bonne nuit
BLEU Score: 100.00/100
-----------------------------------
English:    i am happy
Reference:  je suis heureux
Predicted:  je suis heureux
BLEU Score: 100.00/100
-----------------------------------
English:    i am sad
Reference:  je suis triste
Predicted:  je suis triste
BLEU Score: 100.00/100
-----------------------------------
English:    how are you
Reference:  comment allez vous
Predicted:  comment allez vous
BLEU Score: 100.00/100
-----------------------------------
English:    i love you
Reference:  je t aime
Predicted:  je t aime
BLEU Score: 100.00/100
-------